In [1]:
import pandas as pd
from wikifin_rag.config import PROJECT_ROOT

In [2]:
exchanges_file_path = PROJECT_ROOT / "data" / "evals" / "rag-answers.csv"

In [3]:
sample_exchanges = pd.read_csv(exchanges_file_path)

In [4]:
SAMPLE_QUESTIONS = sample_exchanges.question
SAMPLE_ANSWERS = sample_exchanges.answer_llm
RELEVANCE = ["RELEVANT", "PARTLY_RELEVANT", "IRRELEVANT"]

In [5]:
from wikifin_rag.items import LLMCallRecord
import random
from datetime import datetime

In [6]:
def fake_record(question, answer):
    prompt_tokens = random.randint(50, 200)
    completion_tokens = random.randint(50, 300)
    total_tokens = prompt_tokens + completion_tokens

    input_cost = random.uniform(0.0001, 0.005)
    output_cost = random.uniform(0.0001, 0.005)
    total_cost = input_cost + output_cost

    year = random.choice([2024, 2025, 2026])
    month = random.choice(range(1, 13))
    day = random.choice(range(1, 29))


    return LLMCallRecord(
        model="gpt-5.4-mini",
        prompt=question,
        instructions="",
        answer=answer,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        response_time=random.uniform(0.5, 5.0),
        input_cost=input_cost,
        output_cost=output_cost,
        total_cost=total_cost,
        timestamp=datetime(year=year, month=month, day=day)
    )

In [7]:
def random_score():
    return random.choice([1, 1, 1, 1, 0])

In [8]:
from wikifin_rag.apps.chat.queries import save_conversation, save_exchange, save_feedback
from wikifin_rag.evaluation_utils import map_progress
from concurrent.futures import ThreadPoolExecutor

In [9]:
def get_random_conversation_id():
    db_client = MonitoringDBClient()

    try:
        with db_client.get_db_connection() as con:
            cur = con.execute(
                f"""
                SELECT id
                FROM {db_client.conversations_table_identifier}
                """
            )
            rows = cur.fetchmany(size=100)
    except Exception as e:
        db_client.logger.error(f"Error retrieving the data: {e}")
        raise

    return random.choice(rows)[0]

In [10]:
def generate_feedback_record(exchange):
    question = exchange[0]
    answer = exchange[1]
    
    record = fake_record(question, answer)

    if random.random() < 0.3:
        conversation_id = get_random_conversation_id()
    else:
        conversation_id = save_conversation(record)
    exchange_id = save_exchange(conversation_id=conversation_id, record=record, query=question)

    if random.random() < 0.7:
        relevance = random.choice(RELEVANCE)
        save_feedback(
            exchange_id=exchange_id,
            source="judge",
            relevance=relevance,
            explanation=f"Answer is {relevance.lower()}."
        )

    if random.random() < 0.5:
        score = random_score()
        save_feedback(
            exchange_id=exchange_id,
            source="user",
            score=score
        )

In [11]:
from wikifin_rag.db_client import MonitoringDBClient

In [12]:
db_client = MonitoringDBClient()
db_client.init_db(drop=True)

In [13]:
with ThreadPoolExecutor(max_workers=6) as pool:
    map_progress(pool, list(zip(SAMPLE_QUESTIONS, SAMPLE_ANSWERS)), generate_feedback_record)

  0%|          | 0/200 [00:00<?, ?it/s]